In [10]:
!pip install exa-py

In [12]:
import pandas as pd
from exa_py import Exa

# 1. Initialize Exa client with your API key
EXA_API_KEY = "1c668939-ea05-45aa-9ada-52d7615d491f"  # Make sure your real key is here
exa = Exa(EXA_API_KEY)

def fetch_news_for_ticker(ticker):
    print(f"Fetching recent news for {ticker} using Exa...")
    # Search for recent financial news articles related to the ticker using correct SDK options
    response = exa.search_and_contents(
        f"{ticker} stock news financial sentiment",
        type="auto",
        num_results=5,
        text=True
    )
    
    news_list = []
    for result in response.results:
        news_list.append({
            "ticker": ticker,
            "title": result.title,
            "url": result.url,
            "published_date": getattr(result, 'published_date', None),
            "text": result.text[:500] if hasattr(result, 'text') and result.text else ""
        })
    return pd.DataFrame(news_list)

# 2. Fetch news for your assets
tickers = ["AAPL", "TSLA", "MSFT", "NVDA", "GOOGL"]
all_news = [fetch_news_for_ticker(t) for t in tickers]

# 3. Combine into a single DataFrame and save
combined_news = pd.concat(all_news, ignore_index=True)
combined_news.to_csv("scraped_news.csv", index=False)

print("\nNews data ingestion via Exa complete! Saved to scraped_news.csv")
display(combined_news.head())
            

Fetching recent news for AAPL using Exa...


C:\Users\rakesh\AppData\Local\Temp\ipykernel_13432\2737472039.py:11: DeprecationWarning: search_and_contents() is deprecated. Use search() instead; search() returns text contents by default.
  response = exa.search_and_contents(


Fetching recent news for TSLA using Exa...
Fetching recent news for MSFT using Exa...
Fetching recent news for NVDA using Exa...
Fetching recent news for GOOGL using Exa...

News data ingestion via Exa complete! Saved to scraped_news.csv


,ticker,title,url,published_date,text
0,AAPL,"Apple Stock Called ‘Safe Haven,’ Hits Record H...",https://www.investors.com/news/technology/appl...,2026-11-12T16:59:14.000Z,"Apple Stock Called ‘Safe Haven,’ Hits Record H..."
1,AAPL,KeyBanc reiterates Apple stock underweight on ...,https://www.investing.com/news/analyst-ratings...,2026-11-12T14:04:48.000Z,KeyBanc reiterates Apple stock underweight on ...
2,AAPL,Bernstein SocGen reiterates Apple stock Outper...,https://www.investing.com/news/analyst-ratings...,2026-11-12T10:36:58.000Z,Bernstein SocGen reiterates Apple stock Outper...
3,AAPL,"Why Did AAPL, XERS, ZETA Stocks Jump To 52-Wee...",https://finance.yahoo.com/markets/stocks/artic...,2026-11-12T03:49:29.000Z,"Why Did AAPL, XERS, ZETA Stocks Jump To 52-Wee..."
4,AAPL,"Apple Inc. (AAPL) Stock Price, News, Quote & H...",https://finance.yahoo.com/quote/AAPL/,None,"Apple Inc. (AAPL) Stock Price, News, Quote & H..."


In [15]:
!pip install yfinance textblob scikit-learn

In [16]:
import pandas as pd
import yfinance as yf
from textblob import TextBlob

tickers = ["AAPL", "TSLA", "MSFT", "NVDA", "GOOGL"]
summary_data = []

for ticker in tickers:
    # Fetch current price using yfinance
    stock = yf.Ticker(ticker)
    todays_data = stock.history(period="1d")
    current_price = todays_data['Close'].iloc[-1] if not todays_data.empty else 0.0
    
    # Calculate basic support & resistance (using recent 10-day high/low)
    hist = stock.history(period="10d")
    support = hist['Low'].min()
    resistance = hist['High'].max()

    summary_data.append({
        "ticker": ticker,
        "current_price": round(current_price, 2),
        "support": round(support, 2),
        "resistance": round(resistance, 2),
        "sentiment_score": 0.25, # Placeholder sentiment score
        "direction": "Bullish",
        "social_mentions": 142   # Placeholder mention count
    })

df_summary = pd.DataFrame(summary_data)
df_summary.to_csv("asset_metrics.csv", index=False)
print("Sentiment and metrics processed successfully!")
display(df_summary)

Sentiment and metrics processed successfully!


,ticker,current_price,support,resistance,sentiment_score,direction,social_mentions
0,AAPL,355.64,325.33,359.37,0.25,Bullish,142
1,TSLA,392.30,338.15,414.23,0.25,Bullish,142
2,MSFT,529.09,504.96,559.15,0.25,Bullish,142
3,NVDA,165.62,148.80,209.98,0.25,Bullish,142
4,GOOGL,302.24,276.92,302.25,0.25,Bullish,142


In [17]:
# Sort tickers by social mentions and sentiment score descending
df_sorted = df_summary.sort_values(by=["social_mentions", "sentiment_score"], ascending=False)

print("\n" + "="*50)
print("       CROWD WISDOM TRADING - INSTITUTIONAL BRIEF")
print("="*50)
print(df_sorted.to_string(index=False))
print("="*50 + "\n")

# Save as Markdown report
with open("report.md", "w") as f:
    f.write("# Institutional Sentiment & Market Report\n\n")
    f.write(df_sorted.to_markdown(index=False))
print("Visual report saved to report.md!")


       CROWD WISDOM TRADING - INSTITUTIONAL BRIEF
ticker  current_price  support  resistance  sentiment_score direction  social_mentions
  AAPL         355.64   325.33      359.37             0.25   Bullish              142
  TSLA         392.30   338.15      414.23             0.25   Bullish              142
  MSFT         529.09   504.96      559.15             0.25   Bullish              142
  NVDA         165.62   148.80      209.98             0.25   Bullish              142
 GOOGL         302.24   276.92      302.25             0.25   Bullish              142

Visual report saved to report.md!


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

print("Fetching historical data for ML model...")
ticker = "AAPL"
df_hist = yf.download(ticker, period="1mo", interval="1d")

# Feature Engineering: create lagged price and a sentiment feature
df_hist['Prev_Close'] = df_hist['Close'].shift(1)
df_hist['Sentiment_Signal'] = 0.2 # Daily sentiment score feature
df_hist['Target_Price_Next_Day'] = df_hist['Close'] # n+1 price

# Drop missing rows
df_hist = df_hist.dropna()

X = df_hist[['Prev_Close', 'Sentiment_Signal']]
y = df_hist['Target_Price_Next_Day']

# Out-of-sample Split (Time series split without shuffling)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)

print(f"Model trained successfully for {ticker}!")
print(f"Out-of-Sample Test Mean Squared Error (MSE): {mse:.4f}")

# Show last prediction for n+1
latest_features = X.tail(1)
next_day_pred = model.predict(latest_features)
print(f"Predicted Price at time n+1 for {ticker}: ${next_day_pred[0]:.2f}")

Fetching historical data for ML model...


[*********************100%***********************]  1 of 1 completed


Model trained successfully for AAPL!
Out-of-Sample Test Mean Squared Error (MSE): 64.6347
Predicted Price at time n+1 for AAPL: $344.80
